### In a conveniently named eval.ipynb, we will evaluate and compare the base LLM to our minimally trained model to compare differences.

In [16]:
from unsloth import FastLanguageModel
import torch

In [17]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = 1024,
    load_in_4bit = True
)

FastLanguageModel.for_inference(model);

==((====))==  Unsloth 2026.1.4: Fast Llama patching. Transformers: 4.57.1. vLLM: 0.11.2.
   \\   /|    NVIDIA GeForce RTX 5080 Laptop GPU. Num GPUs = 1. Max memory: 15.92 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


# Instructions that tell the LLM how to use the REPL environment

In [18]:
SYSTEM_PROMPT = """You are a SEARCH assistant with a Python REPL. You search documents - nothing else.

OUTPUT FORMAT: Your response must START with ```python - no preamble, no explanation, just code.

CONSTRAINT: Your training data is IRRELEVANT. You know NOTHING about this document.
- Answering without searching = WRONG
- Explaining instead of searching = WRONG
- Any text before your code block = WRONG

TOOLS:
- `context` - the document (already loaded, DO NOT redefine)
- `llm_query(question, context[start:end])` - ask sub-LLM about a chunk

WORKFLOW:
1. Write ```python with print() to search
2. STOP immediately after code block
3. Wait for output (appears in next message)
4. Search more OR give FINAL(answer)

SEARCH STRATEGY:
- If find() returns -1, try DIFFERENT keywords (not the same one)
- Try 5-10 keywords before concluding something isn't covered
- Use words FROM the question: vessel, ship, foot, horsepower, italic, abbreviate
- Try simpler terms, synonyms, related concepts

DO NOT:
- Explain what you're doing
- Teach Python concepts
- Answer from memory
- Write multiple code blocks
- Add text before ```python

EXAMPLE (your entire response should look like this):
```python
idx = context.find("vessel")
print(f"Found at: {idx}")
if idx != -1:
    print(context[idx:idx+500])
```

When done searching, end with: FINAL(your evidence-based answer)"""

In [19]:
messages = [
    {
        "role": "user",
        "content": SYSTEM_PROMPT + "\n\n---\n\nWhat are the main topics discussed in this document?"
    }
]

In [20]:
input_ids = tokenizer.apply_chat_template(messages, return_tensors = "pt", add_generation_prompt = True).to("cuda")
outputs = model.generate(input_ids, max_new_tokens = 512)
base_output = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens = True)
print(base_output)

```python
def find_topics():
    return context.topics

topics = find_topics()
if not topics:
    print("No topics found in the document.")
else:
    print(topics)
```

---

FINAL(your evidence-based answer)


this is one example output from the above cell: 

```python
import context

def search_topics():
    topics = []
    for keyword in ["vessel", "ship", "foot", "horsepower", "_", "abbreviate"]:
        idx = context.find(keyword)
        if idx!= -1:
            topics.append(keyword)
    return topics

topics = search_topics()
print("Topics:", topics)
```


#### As you can tell, this is crap. Why is context being imported when its a variable? Why is everything wrapped inside of a function? the RLM function's exec(), so while this is technically fine it just makes it unnecessarily complex. Here's what an expected answer looks like:

```python
    idx = context.find("topics")
    print(f"Found at: {idx}")
    if idx != -1:
        print(context[idx:idx+500])
```

### Now let's load in the SFT model

In [24]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = 1024,
    load_in_4bit = True,
)

model.load_adapter("/workspace/outputs/rlm_lora_v2", adapter_name = "default")
FastLanguageModel.for_inference(model)
print(model); # to prove that it is actually the new model

==((====))==  Unsloth 2026.1.4: Fast Llama patching. Transformers: 4.57.1. vLLM: 0.11.2.
   \\   /|    NVIDIA GeForce RTX 5080 Laptop GPU. Num GPUs = 1. Max memory: 15.92 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048, padding_idx=128004)
    (layers): ModuleList(
      (0): LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
            (lora_dropout): ModuleDict(
              (default): Identity()
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=2048, out_featur

#### Now we test again.

In [22]:
inputs_id = tokenizer.apply_chat_template(messages, return_tensors = "pt", add_generation_prompt = True).to("cuda")
outputs = model.generate(inputs_id, max_new_tokens = 512)
base_output = tokenizer.decode(outputs[0][inputs_id.shape[1]:], skip_special_tokens = True)
print(base_output)

```python
idx = context.find("vessel")
print(f"Found at: {idx}")
if idx!= -1:
    print(context[idx:idx+500])
```


#### example output: 

```python
idx = context.find("vessel")
print(f"Found at: {idx}")
if idx!= -1:
    print(context[idx:idx+500])
```

The difference is already tangible. It did good in fitting that flat format, but it's bad in the sense that it literally ran a single check and considered it a day. A good turn would involve multiple keywords, etc. and a more broad search. But this is only turn 1 of a multi-turn REPL loop. In a real example it would keep searching ideally.

#### The system prompt sucks. It was trained on this specific dataset I was working with which told it exactly what to do. But that isn't the biggest factor for now. Let's ask it a different question that is completely different from what the system prompt suggests.

In [23]:
messages = [
    {
        "role": "user",
        "content": SYSTEM_PROMPT + "\n\n---\n\nWho is the author of this text?"
    }
]

#### now we compare. First the base model:

In [25]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = 1024,
    load_in_4bit = True
)

FastLanguageModel.for_inference(model);

input_ids = tokenizer.apply_chat_template(messages, return_tensors = "pt", add_generation_prompt = True).to("cuda")
outputs = model.generate(input_ids, max_new_tokens = 512)
base_output = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens = True)
print(base_output)

==((====))==  Unsloth 2026.1.4: Fast Llama patching. Transformers: 4.57.1. vLLM: 0.11.2.
   \\   /|    NVIDIA GeForce RTX 5080 Laptop GPU. Num GPUs = 1. Max memory: 15.92 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
```python
# No code here, just the author's name
# Assuming it's "author" (the actual name is not provided)
author = "author"  # Replace with the actual name
```


#### example output: 

```python
# No code here, just the author's name
# Assuming it's "author" (the actual name is not provided)
author = "author"  # Replace with the actual name
```

#### Now for the SFT model:

In [26]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = 1024,
    load_in_4bit = True,
)

model.load_adapter("/workspace/outputs/rlm_lora_v2", adapter_name = "default")
FastLanguageModel.for_inference(model)

inputs_id = tokenizer.apply_chat_template(messages, return_tensors = "pt", add_generation_prompt = True).to("cuda")
outputs = model.generate(inputs_id, max_new_tokens = 512)
base_output = tokenizer.decode(outputs[0][inputs_id.shape[1]:], skip_special_tokens = True)
print(base_output)

==((====))==  Unsloth 2026.1.4: Fast Llama patching. Transformers: 4.57.1. vLLM: 0.11.2.
   \\   /|    NVIDIA GeForce RTX 5080 Laptop GPU. Num GPUs = 1. Max memory: 15.92 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
```python
idx = context.find("vessel")
if idx!= -1:
    print(context[idx:idx+500])
```


#### example output:

```python
idx = context.find("vessel")
if idx!= -1:
    print(context[idx:idx+500])
```

### Honestly, it seems bad but it is very impressive for a tiny 1B parameter model to be able to realize that they at least had to run .find() on the author except it searched vessel. It ran flat code, and can print a slice. this can actually work unlike the untrained result. SFT easily worked wonders. So all that's left to do now is to #1, improve the system prompt because it did kind of force it to search certain things, and also run more epochs and examples. I will work on fixing that in sft_8b.ipynb and eval_8b.ipynb.